In [5]:

"""未来函数（look-ahead bias）自检工具，供参赛者自行排查因子是否偷看了未来数据。

用法
----
分别用你的 main() 跑两次，只改 end_date，其余参数（start_date、datasets）保持不变：

    df_full = main(datasets, start_date, end_date)      # 全窗
    df_cut  = main(datasets, start_date, cutoff_date)   # 截断窗，cutoff_date < end_date

    from lookahead_selfcheck import check_lookahead
    check_lookahead(df_full, df_cut, cutoff_date)

原理：因子在算某一天的值时，如果没有偷看未来数据，那么无论后面的日期是否被砍掉，
这一天算出来的值都应该完全一样。所以把 end_date 砍到 cutoff 后再跑一次，
两次结果在 date <= cutoff 的部分理应逐格一致；如果不一致，说明算 cutoff 之前某天的
因子值时用到了 cutoff 之后才有的数据，即未来函数。
"""
from __future__ import annotations

import numpy as np
import pandas as pd

_KEY_COLS = ("date", "instrument")


def check_lookahead(
    df_full: pd.DataFrame,
    df_cut: pd.DataFrame,
    cutoff: str,
    factor_col: str | None = None,
    rtol: float = 1e-5,
    atol: float = 1e-8,
) -> bool:
    """比对全窗口与截断窗口的因子输出，检查是否存在未来函数。

    Args:
        df_full: 全窗口跑 main() 得到的因子数据（含 date/instrument + 因子列）。
        df_cut:  截断窗口跑 main() 得到的因子数据，end_date 换成 cutoff，其余参数不变。
        cutoff:  截断日期 'YYYY-MM-DD'，只比对 date <= cutoff 的部分。
        factor_col: 因子列名，默认自动取 date/instrument 之外的唯一列。
        rtol/atol: 浮点比对容差，语义同 numpy.isclose。

    Returns:
        bool: True 表示未发现未来函数，False 表示发现差异（差异详情已打印）。
    """
    if factor_col is None:
        factor_col = _pick_factor_col(df_full)

    cutoff_ts = pd.Timestamp(cutoff).normalize()
    full = _prep(df_full, factor_col, cutoff_ts)
    cut = _prep(df_cut, factor_col, cutoff_ts)

    merged = pd.merge(full, cut, how="outer", on=list(_KEY_COLS), suffixes=("_full", "_cut"))
    if merged.empty:
        print(f"[未来函数自检] cutoff={cutoff_ts.date()} 前两侧都没有数据，无法判定。")
        return True

    a = merged["factor_full"].to_numpy(dtype=float)
    b = merged["factor_cut"].to_numpy(dtype=float)
    a_nan, b_nan = np.isnan(a), np.isnan(b)
    both_finite = ~a_nan & ~b_nan

    close = np.zeros(len(merged), dtype=bool)
    close[both_finite] = np.isclose(a[both_finite], b[both_finite], rtol=rtol, atol=atol)
    diff_mask = ~close & ~(a_nan & b_nan)

    diff_cells = int(diff_mask.sum())
    if diff_cells == 0:
        print(f"[未来函数自检] 通过：cutoff={cutoff_ts.date()} 前共 {len(merged)} 个格子全部一致，未发现未来函数。")
        return True

    diff_df = merged.loc[diff_mask, ["date", "instrument", "factor_full", "factor_cut"]].copy()
    diff_df["abs_dev"] = (diff_df["factor_full"] - diff_df["factor_cut"]).abs()
    first_diff_date = diff_df["date"].min().date()
    diff_ratio = diff_cells / len(merged)

    print(
        f"[未来函数自检] 发现差异：cutoff={cutoff_ts.date()} 前 {diff_cells}/{len(merged)} "
        f"({diff_ratio:.2%}) 个格子不一致，最早差异出现在 {first_diff_date}，"
        f"说明算 {first_diff_date} 这天的因子值时用到了 cutoff 之后才有的数据。"
    )
    print(diff_df.sort_values(["date", "abs_dev"], ascending=[False, False]).head(20).to_string(index=False))
    return False


def _pick_factor_col(df: pd.DataFrame) -> str:
    cols = [c for c in df.columns if c not in _KEY_COLS]
    if not cols:
        raise ValueError("未找到因子列（date/instrument 之外）")
    return cols[0]


def _prep(df: pd.DataFrame, factor_col: str, cutoff_ts: pd.Timestamp) -> pd.DataFrame:
    out = df[[*_KEY_COLS, factor_col]].copy()
    out["date"] = pd.to_datetime(out["date"]).dt.normalize()
    out["instrument"] = out["instrument"].astype(str)
    out = out.rename(columns={factor_col: "factor"})
    out["factor"] = pd.to_numeric(out["factor"], errors="coerce")
    return out[out["date"] <= cutoff_ts]


In [6]:
def main(datasources, start_date, end_date):
    """
    因子构建主函数（演示：带未来函数 / 前视偏差的极简因子）

    评测时平台会自动替换 datasources / start_date / end_date 三个入参并调用本函数

    参数:
        datasources (dict): 数据源表名映射 {逻辑名: 物理表名}
                                "bar1m"     -> 分钟 K 线表
                                "financial" -> 财务数据表
        start_date (str): 开始时间
        end_date (str):   结束时间

    返回:
        pd.DataFrame: 因子数据，须包含三列 ['date', 'instrument', 'factor']，且不含 inf
    """
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]

    # 用未来函数，需要向后多取若干天，才能算出区间末尾几天的"明日收益"
    LOOKAHEAD_DAYS = 7
    query_end_date = pd.to_datetime(end_date) + pd.Timedelta(days=LOOKAHEAD_DAYS)

    # ===== 编写因子 SQL =====
    # 【未来函数示例】把"明日日收益率"直接当作"今日因子值"。
    # 由于评测用的是 T+1 收益，因子几乎等于被预测目标本身，IC 会异常地高（接近 1），
    # 这正是典型的前视偏差（look-ahead bias），仅用于演示，切勿用于实盘/正式提交。
    sql = f"""
    WITH cte_daily AS (
        -- 分钟 K 线聚合成日频收盘价（取每个交易日最后一分钟的 close）
        SELECT
            instrument,
            strftime(date, '%Y-%m-%d') AS trading_day,
            last(close ORDER BY date)  AS close
        FROM {bar1m}
        WHERE close > 0
        GROUP BY instrument, strftime(date, '%Y-%m-%d')
    ),
    cte_lead AS (
        SELECT
            *,
            -- 关键的未来函数：lead 取到"下一交易日"的收盘价
            lead(close, 1) OVER (PARTITION BY instrument ORDER BY trading_day) AS next_close
        FROM cte_daily
    )
    SELECT
        CAST(trading_day AS DATETIME) AS date,
        instrument,
        -- 用明日收益率作为今日因子（未来函数）
        (next_close / close - 1) AS factor
    FROM cte_lead
    WHERE next_close IS NOT NULL
    """

    # 注意：因为要用到 end_date 之后的数据算"明日收益"，这里把查询上界放宽到 query_end_date，
    # 最终再用 start_date ~ end_date 过滤输出。
    df = dai.query(
        sql,
        filters={'date': [start_date, query_end_date.strftime('%Y-%m-%d %H:%M:%S')]},
        compression=True,
    ).df()
    df = df[(df['date'] >= pd.to_datetime(start_date)) & (df['date'] <= pd.to_datetime(end_date))]

    # ===== 对齐股票池 =====
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
    ).df()
    df = pd.merge(df, stk_pool, how='inner', on=['date', 'instrument'])

    return df


if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog
    import time
    import pandas as pd

    # 本地自测时自行构造数据源映射（评测时由平台注入，逻辑名固定为 "bar1m"/"financial"）
    datasources = {'bar1m': 'bigalpha_2026_stock_bar1m_selftest'}
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-10-31 23:59:59'

    # 计算因子
    print(f"计算因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    # 计算截断数据
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-01-31 23:59:59'

    # 只改 end_date 为截断日，其余入参与全窗完全一致；不跑 eval，直接把因子输出落盘。
    _t0 = time.perf_counter()
    raw_cut = main({"bar1m": "bigalpha_2026_stock_bar1m_ahead"}, start_date, end_date)
    print(f"[judge_runner] main() 截断窗耗时 {time.perf_counter() - _t0:.2f}s", flush=True)

    # 强制把返回数据裁到目标检验窗口：
    # 用户 main 可能没按 end_date 严格过滤输出（如把查询上界放宽了 buffer、或压根没裁），
    # 导致 raw_cut 含窗口外的行。检测只比对 date <= cutoff 的格子，窗口外的行是噪声，
    # 且会让存档的校验数据超出「用截断表跑 March」的语义，故在此按 date 硬裁到目标窗口。
    _date = pd.to_datetime(raw_cut["date"]).dt.normalize()
    _lo = pd.Timestamp(start_date).normalize()
    _hi = pd.Timestamp(end_date).normalize()
    raw_cut = raw_cut[(_date >= _lo) & (_date <= _hi)]

    check_lookahead(
        df_full=factor_data,
        df_cut=raw_cut,
        cutoff='2025-01-31 23:59:59'
    )

计算因子，区间：2024-01-01 00:00:00 ~ 2024-10-31 23:59:59
[judge_runner] main() 截断窗耗时 0.34s
[未来函数自检] 发现差异：cutoff=2025-01-31 前 176843/197819 (89.40%) 个格子不一致，最早差异出现在 2024-01-31，说明算 2024-01-31 这天的因子值时用到了 cutoff 之后才有的数据。
      date instrument  factor_full  factor_cut  abs_dev
2024-10-30  000012.SZ     0.014733         NaN      NaN
2024-10-30  000016.SZ     0.100000         NaN      NaN
2024-10-30  000019.SZ    -0.001466         NaN      NaN
2024-10-30  000025.SZ     0.048615         NaN      NaN
2024-10-30  000028.SZ    -0.000717         NaN      NaN
2024-10-30  000029.SZ     0.100000         NaN      NaN
2024-10-30  000030.SZ     0.041339         NaN      NaN
2024-10-30  000034.SZ     0.060870         NaN      NaN
2024-10-30  000035.SZ     0.049145         NaN      NaN
2024-10-30  000048.SZ    -0.017807         NaN      NaN
2024-10-30  000049.SZ    -0.003954         NaN      NaN
2024-10-30  000058.SZ     0.018518         NaN      NaN
2024-10-30  000059.SZ     0.011299         NaN      NaN
2024-10